# Data Preparation: Feature Store with Feast

This notebook mirrors the **Data Preparation** guide at [learnmlops.ops4life.com/guides/data-preparation/](https://learnmlops.ops4life.com/guides/data-preparation/).

Demonstrate how a feature store centralises feature definitions so training and serving use identical transformations. This notebook uses Feast with a local Parquet file source — the same API you'd use in production with S3 or a data warehouse.

**What we'll cover:**
1. What is a feature store and why does it matter
2. Write a feature Parquet file
3. Define Feast entities and feature views
4. Fetch historical features for training
5. Fetch online features for serving (simulated)

In [ ]:
import pandas as pd
import numpy as np
import os
import tempfile

os.makedirs('/workspace/pipeline-data', exist_ok=True)

## Step 1: What is a feature store and why does it matter

Without a feature store, training and serving code often compute features independently. Even small implementation differences (rounding, NULL handling, timezone offsets) cause **training-serving skew** — the model behaves differently in production than in experiments.

A feature store solves this by:
1. **Defining features once** in a central registry
2. **Serving historical features** for offline training (point-in-time correct)
3. **Serving live features** for online inference from a low-latency store

Feast is the most widely adopted open-source feature store. In production the `FileSource` below would point to S3 or GCS; in this sandbox it points to a local Parquet file.

## Step 2: Write a feature Parquet file

In [ ]:
# Load featured data (or generate if pipeline hasn't run yet)
featured_path = '/workspace/pipeline-data/featured.csv'
if os.path.exists(featured_path):
    df = pd.read_csv(featured_path)
    print(f'Loaded featured.csv: {df.shape}')
else:
    np.random.seed(42)
    n = 1470
    df = pd.DataFrame({
        'EmployeeNumber':    np.arange(1, n + 1),
        'OverallSatisfaction': np.random.randint(1, 4, n),
        'AnnualIncome':      np.random.randint(0, 5, n),
        'AgeGroup':          np.random.randint(1, 5, n),
        'PromotionStagnation': np.random.uniform(0, 1, n).round(3),
        'CareerVelocity':    np.random.uniform(0, 1, n).round(3),
        'LongCommute':       np.random.choice([0, 1], n),
        'StableManager':     np.random.choice([0, 1], n),
        'Attrition':         np.random.choice([0, 1], n, p=[0.84, 0.16]),
    })
    print(f'Generated fallback dataset: {df.shape}')

# Add event_timestamp required by Feast for point-in-time joins
df['event_timestamp'] = pd.Timestamp('2025-01-01', tz='UTC')
df['employee_number'] = df['EmployeeNumber'] if 'EmployeeNumber' in df.columns else np.arange(1, len(df) + 1)

parquet_path = '/workspace/pipeline-data/employee_features.parquet'
df.to_parquet(parquet_path, index=False)
print(f'Written: {parquet_path}')

## Step 3: Define Feast entities and feature views

In [ ]:
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64, String
from datetime import timedelta

# Entity: the primary key that identifies each employee
employee = Entity(
    name="employee",
    description="An employee identified by their employee number",
)

# Source: local Parquet file (replace path with s3://... in production)
employee_source = FileSource(
    path=parquet_path,
    event_timestamp_column="event_timestamp",
)

# Feature view: a set of features for the employee entity
attrition_features = FeatureView(
    name="employee_attrition_features",
    entities=[employee],
    ttl=timedelta(days=365),
    schema=[
        Field(name="OverallSatisfaction", dtype=Int64),
        Field(name="AnnualIncome",        dtype=Int64),
        Field(name="AgeGroup",            dtype=Int64),
        Field(name="PromotionStagnation", dtype=Float32),
        Field(name="CareerVelocity",      dtype=Float32),
        Field(name="LongCommute",         dtype=Int64),
        Field(name="StableManager",       dtype=Int64),
    ],
    source=employee_source,
)

print("Feast objects defined:")
print(f"  Entity:       {employee.name}")
print(f"  FeatureView:  {attrition_features.name}  ({len(attrition_features.schema)} features)")

In production you would run `feast apply` to register these definitions in a feature registry (backed by a database or cloud storage). The registry makes them discoverable by any team or service.

## Step 4: Fetch historical features for training

In [ ]:
# Point-in-time correct join: for each entity + timestamp, fetch the feature values
# that were valid at that exact moment — preventing future leakage
labels_df = pd.DataFrame({
    'employee_number': df['employee_number'].head(5).values,
    'event_timestamp': pd.Timestamp('2025-01-01', tz='UTC'),
    'Attrition': df['Attrition'].head(5).values,
})

# Simulate what store.get_historical_features() would return
feature_cols = ['OverallSatisfaction', 'AnnualIncome', 'AgeGroup',
                'PromotionStagnation', 'CareerVelocity', 'LongCommute', 'StableManager']
training_df = df.head(5)[['employee_number', 'event_timestamp'] + feature_cols].copy()

print("Training features (first 5 rows):")
print(training_df)

In a live Feast deployment this would be:
```python
store = FeatureStore(repo_path="./feature_repo")
training_df = store.get_historical_features(
    entity_df=labels_df,
    features=["employee_attrition_features:OverallSatisfaction", ...],
).to_df()
```
Feast performs a point-in-time join so each label row gets the feature values that existed *before* its timestamp — preventing leakage from future data.

## Step 5: Fetch online features for serving (simulated)

In [ ]:
# In production: store.get_online_features(features=[...], entity_rows=[{"employee_number": 42}])
# Here we simulate the lookup from the Parquet source
employee_number = 42
row = df[df['employee_number'] == employee_number][feature_cols].iloc[0]

online_features = row.to_dict()
print(f"Online features for employee {employee_number}:")
for k, v in online_features.items():
    print(f"  {k}: {v}")

## Next Steps

- Train a model using these features: `02-pipeline/ml-model-training/train.ipynb`
- Full guide: [learnmlops.ops4life.com/guides/data-preparation/](https://learnmlops.ops4life.com/guides/data-preparation/)